┌─────────────────────────────────────────────────────────┐
│                    DATA LAYER                           │
│  (Your existing pipeline — historical + live streaming) │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  FEATURE LAYER                          │
│  (Alignment, resampling, all engineered features)       │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  SIGNAL LAYER                           │
│  (Individual setups A/B/C/D — each independently)      │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  RISK LAYER                             │
│  (Position sizing, max drawdown, correlation limits)    │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│               EXECUTION LAYER                           │
│  (Order routing, slippage, Binance API)                 │
└─────────────────────────────────────────────────────────┘

In [ ]:
from orderflow import futures_agg_trades, get_oi, get_mark_price_klines, spot_agg_trades, get_premium_index_klines
import pandas as pd

symbol = "BTCUSDT"
date = "2026-06-12"

fut_trades = futures_agg_trades(symbol, date)
oi = get_oi(symbol, date)
context = get_mark_price_klines(symbol, date)
spot = spot_agg_trades(symbol, date)
premium = get_premium_index_klines(symbol, date)

mark_5m = context.resample('5min').agg({"open":'first',"high":"max", "low":"min", "close":"last"})

premium_5min = premium["close"].resample("5min").last()
R = 0.0001
funding_raw = premium_5min + R
funding_5min = funding_raw.clip(-0.0005, 0.0005)
funding_5min.name = 'funding_rate'
funding_5min = pd.DataFrame(funding_5min)
funding_5min["8h_avg"] = funding_5min["funding_rate"].rolling("0.5h").mean()
combined = pd.concat([fut_trades['fut_cumulative_volume_delta'], spot['spot_cumulative_volume_delta'], mark_5m, oi['sum_open_interest'], funding_5min], axis=1)

                         open      high       low     close     volume  \
open_time                                                                
2026-06-12 00:05:00  63583.99  63810.01  63583.99  63718.65  443.03552   
2026-06-12 00:10:00  63718.65  63739.83  63684.30  63736.58   34.37345   
2026-06-12 00:15:00  63736.57  63736.58  63619.30  63642.00   23.43065   
2026-06-12 00:20:00  63642.00  63676.00  63582.00  63582.00   23.08913   
2026-06-12 00:25:00  63582.01  63636.86  63520.86  63616.00   27.95399   
...                       ...       ...       ...       ...        ...   
2026-06-12 23:35:00  63443.00  63508.91  63443.00  63483.44  116.04415   
2026-06-12 23:40:00  63483.45  63569.99  63483.45  63505.01   19.04476   
2026-06-12 23:45:00  63505.01  63536.00  63505.01  63532.88   13.43945   
2026-06-12 23:50:00  63532.89  63570.32  63532.88  63570.00    8.04212   
2026-06-12 23:55:00  63570.01  63580.01  63562.00  63580.01   27.35119   

                           close_time

In [3]:
from orderflow import build_order_flow_chart

my_layout = [
    {"type": "candlestick", "title": "Price Action", "name": "Candlestick", "y_title": "Price (USDT)"},
    {"type": "line", "title": "Open Interest", "column": "sum_open_interest", "name": "OI Coins", "color": "purple", "y_title": "OI"},
    {"type": "delta_bars", "title": "Funding Rate", "column": "funding_rate", "name": "Funding", "color": "orange", "y_title": "Rate"},
    {"type": "delta_bars", "title": "Spot CVD", "column": "spot_cumulative_volume_delta", "name": "Spot Delta", "y_title": "Delta"},
    {"type": "delta_bars", "title": "Futures CVD", "column": "fut_cumulative_volume_delta", "name": "Futures Delta", "y_title": "Delta"}
]

# Generate a 5-panel chart completely automatically!
fig = build_order_flow_chart(combined, "2026-06-14", config=my_layout)
fig.show()